# Supplementary Figure 11

Pathway-level cross-omics profiles and sensitivity analyses (v30). Panels a and b show how well distances between patients (PREDICT, 77 patients) or participant-by-group profiles (CapScan, 13 participants with all five sampling groups) agree between molecular layers for KEGG pathway profiles; the native, eggNOG OG and KO matrices are shown in Fig. 4i and 5g and are no longer drawn here. Panels c and d compare native correlations with all observed features and with a 10% prevalence filter, and panels e and f vary the distance and correlation method. Every coefficient, P value and q value is also recomputed from the distance matrices.

Run from the repository root: `python paper.py build SFig11`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

In [ ]:
from IPython.display import Image, display
import pandas as pd
FIG5 = REPO / "source_data" / "Fig5"      # CapScan rows and CapScan functional distances (shared with Figure 5g)
inputs = panels.load_inputs(SD, FIG5)       # the three tables the original producer read
print({k: v.shape for k, v in inputs.items()})

## Panel a

PREDICT KEGG pathway distance-correlation matrix, all observed features. Supports "pathway-level RNA–protein agreement was weaker (ρ=0.16, q=0.024; Supplementary Fig. 11a and Source Data)" (v30 paragraph 74) and the legend (v30 paragraph 241). n = 77 patients. Test: Spearman correlation between the upper triangles of two Euclidean distance matrices, two-sided permutation test with 1,999 patient-label permutations, Benjamini–Hochberg within the complete functional family (28 comparisons, OG/KO/pathway). Asterisks: q < 0.05. All six panels are drawn on one canvas by the v30 build; this cell renders it once (`build/SFig11/SFig11_v30.png`, `.pdf`).

In [ ]:
plotted = panels.render(inputs, OUT / "SFig11_v42")
display(Image(filename=str(OUT / "SFig11_v42.png"), width=700))

## Panel b

CapScan KEGG pathway matrix, as panel a. Profiles are participant-by-group cells (13 participants x 5 sampling groups = 65 profiles), adjusted for sampling-group means. Supports the legend (v30 paragraph 241) and Methods (v30 paragraph 427: Benjamini–Hochberg "jointly across all 16 functional-profile comparisons: OG/KO in Fig. 5g and pathways in Supplementary Fig. 11b"). n = 13 participants. Test as in panel a, with 1,999 whole-participant block permutations. Drawn on the canvas rendered under Panel a.

## Panel c

PREDICT native-profile correlations with all observed features (blue) and with the ≥10% prevalence filter (orange). Supports the legend (v30 paragraph 241) and Supplementary Methods (v30 paragraph 416). n = 77 patients. Test: as panel a, 10 comparisons per family. Drawn on the canvas rendered under Panel a.

## Panel d

CapScan native-profile correlations, all observed versus ≥10% prevalence. n = 13 participants (65 profiles). Test: as panel b, 6 comparisons per family. Drawn on the canvas rendered under Panel a.

## Panel e

PREDICT controlled native-distance grid. M1–M4 use the ≥10% prevalence filter, M5–M8 all observed features; within each set Hellinger/Pearson, Hellinger/Spearman, rank/Pearson, rank/Spearman. Supports the legend (v30 paragraph 241). n = 77 patients. Test: outer Pearson or Spearman correlation of distance-matrix upper triangles, 1,999 patient-label permutations, Benjamini–Hochberg within each method family of 10 comparisons. Drawn on the canvas rendered under Panel a.

## Panel f

CapScan controlled grid on region-adjusted profiles. Supports "All Hellinger-distance comparisons passed multiple-testing correction, whereas 9 of 24 rank-based comparisons did not, mostly those involving metabolites (Supplementary Fig. 11f and Source Data)" (v30 paragraph 83) and the counts in Supplementary Methods (v30 paragraph 421). n = 13 participants (65 profiles). Test: as panel e with whole-participant block permutations, 6 comparisons per family.

In [ ]:
pd.DataFrame([panels.stat_panel_f_counts(inputs["grid"])])

## Statistics

Every plotted coefficient, P value and q value is recomputed from the distance matrices with the upstream test code (seed 20260907, 1,999 permutations) and compared with the producer tables. The plotted values are compared with the v30 build's plotted-values table.

In [ ]:
rec = panels.recompute_all(SD, FIG5)
cmp = panels.compare_to_inputs(rec, inputs)
display(cmp)
assert (cmp.max_abs_difference < 1e-12).all() and (cmp.q_class_changes == 0).all()
plotted.to_csv(OUT / "plotted_values_v30.tsv", sep="\t", index=False)
print("plotted values byte-identical to the v30 build:",
      (OUT / "plotted_values_v30.tsv").read_bytes() == (UNIT / "expected" / "plotted_values_v30.tsv").read_bytes(), len(plotted), "values")
import numpy, scipy, statsmodels
versions = f"numpy {numpy.__version__}; scipy {scipy.__version__}; statsmodels {statsmodels.__version__}"
st = panels.stats_table(rec, versions)
st.to_csv(OUT / "STATS.tsv", sep="\t", index=False)
print("STATS.tsv reproduced:", (OUT / "STATS.tsv").read_bytes() == (UNIT / "STATS.tsv").read_bytes(), len(st), "tests")

In [ ]:
from tools.check_pixels import compare
if not fl._HAVE_STYLE:
    print("pixel checks SKIPPED: fonts absent")
else:
    for new, exp_, stem in [(OUT / "SFig11_v42.png", UNIT / "expected" / "SFig11_v42.png", "SFig11_v42"),
                            (OUT / "SFig11_v42.pdf", UNIT / "expected" / "SFig11_v42.pdf", "SFig11_v42_pdf")]:
        r = compare(new, exp_, UNIT / "checks" / stem, dpi=300)
        print(stem, r["verdict"], "differing pixels:", r.get("differing_pixels"))
        assert r["verdict"] == "IDENTICAL", r

l74 = panels.stat_line139(rec["functional"])
f = panels.stat_panel_f_counts(rec["controlled"])
ret = panels.stat_retention(pd.read_csv(SD / "predict_native_retained_signal.tsv", sep="\t", float_precision="round_trip"))
d = panels.stat_design(inputs, rec)
W = panels.NUMBER_WORDS
numbers = [
    ("74", "rho pathway metaT vs microbial_metaP", repr(l74["rho"])),
    ("74", "q pathway metaT vs microbial_metaP", repr(l74["q"])),
    ("83", "region-adjusted rank comparisons not significant", f["rank_not_significant"]),
    ("83", "region-adjusted rank comparisons", f["rank_n"]),
    ("241", "PREDICT patients", d["predict_patients"]),
    ("241", "CapScan participants", W[d["capscan_participants"]]),
    ("241", "sampling groups per participant", W[d["capscan_groups_per_participant"]]),
    ("241", "permutations (1/min P - 1)", d["permutations_from_min_p"]),
    ("241", "controlled estimates", d["controlled_estimates"]),
    ("416", "median retained DNA (%)", repr(100 * ret["DNA"])),
    ("416", "median retained RNA (%)", repr(100 * ret["RNA"])),
    ("421", "region-adjusted rank comparisons not significant", f["rank_not_significant"]),
    ("421", "region-adjusted rank comparisons", f["rank_n"]),
    ("421", "not significant, non-host protein-metabolites", W[f["rank_not_significant_by_pair"]["Non-host protein–Metabolites"]]),
    ("421", "not significant, host protein-metabolites", W[f["rank_not_significant_by_pair"]["Host protein–Metabolites"]]),
    ("421", "not significant, DNA-metabolites", W[f["rank_not_significant_by_pair"]["DNA–Metabolites"]]),
    ("421", "not significant, DNA-host protein", W[f["rank_not_significant_by_pair"]["DNA–Host protein"]]),
    ("421", "region-adjusted Hellinger comparisons significant", f["hellinger_significant"]),
    ("427", "CapScan participants", W[d["capscan_participants"]]),
    ("427", "permutations (1/min P - 1)", d["permutations_from_min_p"]),
    ("427", "CapScan functional family size", d["capscan_functional_family"]),
]
for line, what, value in numbers:
    print(line, what, value, sep="\t")